# Rice Supply Exposure Bubble Chart: 2024 Trade and 2026 Production Gaps

Notebook ini membuat bubble chart untuk empat negara ASEAN yang diproyeksikan memiliki *production shortfall* beras pada 2026: Brunei Darussalam, Malaysia, Philippines, dan Singapore.

Setiap bubble menunjukkan tiga dimensi: 

- **X-axis:** supplier concentration, HHI from UN Comtrade 2024.
- **Y-axis:** projected production shortfall from AFSIS 2026.
- **Bubble area:** rice-import volume from UN Comtrade 2024.

Bubble chart ini adalah perbandingan deskriptif lintas dua tahun data. Nilainya tidak digunakan untuk memprediksi gangguan pasokan atau membentuk kategori risiko resmi.

## 1. Imports and visual settings

The visual style follows the existing supplier-concentration and clustering notebooks: clean white background, light gridlines, direct labels, and high-resolution PNG and SVG exports.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.titleweight': 'bold',
    'figure.dpi': 120,
    'savefig.dpi': 300,
})

TEXT = '#273043'
MUTED = '#5D6470'
GRID = '#E6E8EC'
CORAL = '#D95D39'       # Above the four-country HHI median
TEAL = '#2A9D8F'        # At or below the four-country HHI median
REFERENCE = '#7A828E'
WHITE = '#FFFFFF'

## 2. File configuration

This notebook reuses the exported outputs from the two source notebooks instead of recalculating their full pipelines:

- `ASEAN_Rice_Clustering_Analysis_added.ipynb` exports the AFSIS 2026 master table.
- `rice_import_stacked_bar_2024.ipynb` exports the 2024 HHI summary.

Run those source notebooks first if the CSV files are not yet present.

In [ ]:
MASTER_FILENAME = "asean_master_dataframe.csv"
HHI_FILENAME = "hhi_summary_2024.csv"

def find_project_root():
    """Temukan akar proyek dari folder kerja notebook saat ini."""
    start = Path.cwd().resolve()
    for folder in (start, *start.parents):
        if (folder / "00_data").exists() and (folder / "01_notebooks").exists():
            return folder
    raise FileNotFoundError("Folder akar proyek RASIO tidak ditemukan.")


PROJECT_ROOT = find_project_root()
RAW_DATA_DIR = PROJECT_ROOT / "00_data" / "raw"

MASTER_PATH = PROJECT_ROOT / "00_data" / "processed" / "clustering" / MASTER_FILENAME
HHI_PATH = PROJECT_ROOT / "00_data" / "processed" / "supplier_concentration" / HHI_FILENAME
DATA_OUTPUT_DIR = PROJECT_ROOT / "00_data" / "processed" / "supply_exposure"
FIGURE_DIR = PROJECT_ROOT / "02_visualizations" / "final"
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

DEFICIT_COUNTRIES = [
    "Brunei Darussalam",
    "Malaysia",
    "Philippines",
    "Singapore",
]

print("AFSIS master table:", MASTER_PATH.resolve())
print("UN Comtrade HHI table:", HHI_PATH.resolve())
print("Data output directory:", DATA_OUTPUT_DIR.resolve())
print("Figure directory:", FIGURE_DIR.resolve())


## 3. Load and validate the source outputs

The AFSIS table provides the 2026 projected production shortfall. The trade table provides the 2024 supplier HHI, effective suppliers, top supplier, and import volume.

In [ ]:
master = pd.read_csv(MASTER_PATH)
hhi = pd.read_csv(HHI_PATH)

required_master = {
    'Country', 'SSR_pct', 'Production_Shortfall_pct',
    'Beginning_Stock_to_Use_pct', 'Has_AFSIS_2026', 'Has_Trade_2024',
}
required_hhi = {
    'reporterDesc', 'HHI', 'HHI_10000', 'effective_suppliers',
    'import_tonnes', 'top_supplier', 'top_supplier_share_pct',
}

missing_master = required_master.difference(master.columns)
missing_hhi = required_hhi.difference(hhi.columns)
assert not missing_master, f'Missing master columns: {sorted(missing_master)}'
assert not missing_hhi, f'Missing HHI columns: {sorted(missing_hhi)}'

assert master['Country'].is_unique, 'Country must be unique in the AFSIS master table.'
assert hhi['reporterDesc'].is_unique, 'Reporter must be unique in the HHI summary table.'

print(f'AFSIS rows: {len(master):,}')
print(f'HHI rows: {len(hhi):,}')
display(master.loc[master['Country'].isin(DEFICIT_COUNTRIES), [
    'Country', 'SSR_pct', 'Production_Shortfall_pct',
    'Beginning_Stock_to_Use_pct', 'Has_AFSIS_2026', 'Has_Trade_2024',
]].sort_values('Country'))

## 4. Build the four-country bubble-chart dataset

The population is defined before visualisation: ASEAN members with a positive projected production shortfall and complete bilateral-trade coverage. This produces the four countries used in the import-exposure analysis.

In [ ]:
trade_columns = [
    'reporterDesc', 'HHI', 'HHI_10000', 'effective_suppliers',
    'import_tonnes', 'top_supplier', 'top_supplier_share_pct',
]
trade = hhi[trade_columns].rename(columns={'reporterDesc': 'Country'})

bubble = (
    master.merge(trade, on='Country', how='inner', validate='one_to_one')
    .query('Production_Shortfall_pct > 0')
    .copy()
)

bubble = bubble.loc[bubble['Country'].isin(DEFICIT_COUNTRIES)].copy()
bubble = bubble.sort_values('HHI_10000').reset_index(drop=True)

assert set(bubble['Country']) == set(DEFICIT_COUNTRIES), (
    'The bubble chart must contain exactly the four deficit countries with trade data.'
)
assert len(bubble) == 4, 'Expected four countries in the bubble chart.'
assert bubble[['HHI_10000', 'effective_suppliers', 'import_tonnes', 'Production_Shortfall_pct']].notna().all().all()
assert (bubble['import_tonnes'] > 0).all()

bubble['HHI_reference_group'] = np.where(
    bubble['HHI_10000'] > bubble['HHI_10000'].median(),
    'Above four-country median',
    'At or below four-country median',
)
bubble['Bubble_colour'] = np.where(
    bubble['HHI_reference_group'].eq('Above four-country median'),
    CORAL,
    TEAL,
)

audit_columns = [
    'Country', 'SSR_pct', 'Production_Shortfall_pct',
    'Beginning_Stock_to_Use_pct', 'HHI_10000', 'effective_suppliers',
    'top_supplier', 'top_supplier_share_pct', 'import_tonnes',
]
display(bubble[audit_columns].sort_values('HHI_10000', ascending=False).round({
    'SSR_pct': 1, 'Production_Shortfall_pct': 1,
    'Beginning_Stock_to_Use_pct': 1, 'HHI_10000': 0,
    'effective_suppliers': 1, 'top_supplier_share_pct': 1,
    'import_tonnes': 0,
}))

## 5. Quality checks and descriptive reference lines

The expected HHI values verify that the notebook is using the same supplier-concentration calculations as `rice_import_stacked_bar_2024.ipynb`. The dashed lines in the chart are four-country medians. They are reference lines only, not official food-risk thresholds.

In [ ]:
expected_hhi = {
    'Philippines': 5781.70,
    'Brunei Darussalam': 4769.07,
    'Singapore': 2912.23,
    'Malaysia': 2703.70,
}

actual_hhi = bubble.set_index('Country')['HHI_10000'].to_dict()
for country, expected in expected_hhi.items():
    assert np.isclose(actual_hhi[country], expected, atol=0.10), (
        f'Unexpected HHI for {country}: {actual_hhi[country]:.2f} vs {expected:.2f}'
    )

hhi_median = bubble['HHI_10000'].median()
shortfall_median = bubble['Production_Shortfall_pct'].median()

print('HHI quality check passed.')
print(f'Four-country median HHI: {hhi_median:,.2f}')
print(f'Four-country median production shortfall: {shortfall_median:.2f}%')
print('Coverage: 4 of 10 assessed ASEAN members have a projected production gap.')

## 6. Create the bubble chart

Bubble area uses a square-root transformation of import volume so Brunei remains visible while the Philippines is still shown as the largest importer. Colour is paired with direct labels: coral marks countries above the four-country median HHI and blue-green marks countries at or below it.

In [ ]:
def bubble_area(import_tonnes, minimum_radius=9, maximum_radius=31):
    normalized = np.sqrt(import_tonnes / import_tonnes.max())
    radius = minimum_radius + normalized * (maximum_radius - minimum_radius)
    return np.pi * radius ** 2

label_offsets = {
    'Singapore': (10, -4, 'left', 'top'),
    'Brunei Darussalam': (10, 8, 'left', 'bottom'),
    'Malaysia': (10, 0, 'left', 'center'),
    'Philippines': (-12, 2, 'right', 'bottom'),
}

short_labels = {
    'Brunei Darussalam': 'Brunei',
    'Malaysia': 'Malaysia',
    'Philippines': 'Philippines',
    'Singapore': 'Singapore',
}

fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor(WHITE)
ax.set_facecolor(WHITE)

x_span = bubble['HHI_10000'].max() - bubble['HHI_10000'].min()
ax.set_xlim(
    bubble['HHI_10000'].min() - x_span * 0.17,
    bubble['HHI_10000'].max() + x_span * 0.17,
)
ax.set_ylim(-2, 112)

ax.grid(color=GRID, linewidth=0.9)
ax.set_axisbelow(True)
for side in ['top', 'right']:
    ax.spines[side].set_visible(False)
ax.spines['left'].set_color(GRID)
ax.spines['bottom'].set_color(GRID)

ax.axvline(hhi_median, color=REFERENCE, linewidth=1.2, linestyle=(0, (4, 4)), zorder=1)
ax.axhline(shortfall_median, color=REFERENCE, linewidth=1.2, linestyle=(0, (4, 4)), zorder=1)
ax.text(
    hhi_median + x_span * 0.025, 108, 'Four-country median',
    fontsize=9, color=MUTED, va='top'
)

ax.scatter(
    bubble['HHI_10000'],
    bubble['Production_Shortfall_pct'],
    s=bubble_area(bubble['import_tonnes']),
    c=bubble['Bubble_colour'],
    alpha=0.84,
    edgecolors=WHITE,
    linewidths=2.0,
    zorder=3,
)

for _, row in bubble.iterrows():
    dx, dy, ha, va = label_offsets[row['Country']]
    label = f"{short_labels[row['Country']]}\n{row['effective_suppliers']:.1f} effective suppliers"
    ax.annotate(
        label,
        xy=(row['HHI_10000'], row['Production_Shortfall_pct']),
        xytext=(dx, dy),
        textcoords='offset points',
        ha=ha, va=va,
        fontsize=10, color=TEXT, fontweight='bold',
        zorder=4,
    )

ax.set_xlabel('Supplier concentration, HHI (2024)', fontsize=12, color=TEXT, labelpad=10)
ax.set_ylabel('Projected production shortfall (2026)', fontsize=12, color=TEXT, labelpad=10)
ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value:,.0f}'))
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value:.0f}%'))
ax.tick_params(axis='both', colors=TEXT, labelsize=10)

ax.set_title(
    'Which Deficit Countries Face the Narrowest Import Options?',
    loc='left', fontsize=19, color=TEXT, pad=46, fontweight='bold',
)
ax.text(
    0, 1.075,
    'Supplier concentration, production shortfall, and rice-import volume',
    transform=ax.transAxes, fontsize=11, color=MUTED, ha='left'
)
ax.text(
    0, 1.025,
    '4 of 10 assessed ASEAN members have a projected production gap',
    transform=ax.transAxes, fontsize=10, color=CORAL, ha='left', fontweight='bold'
)

ax.text(
    1, -0.16,
    'Bubble area = 2024 rice-import volume. Dashed lines = four-country medians, not official risk thresholds.',
    transform=ax.transAxes, fontsize=8.8, color=MUTED, ha='right'
)
fig.text(
    0.125, 0.015,
    'Sources: AFSIS 2026 projections; UN Comtrade HS 1006 rice imports by net weight, 2024. Mixed-year descriptive comparison.',
    fontsize=8.5, color=MUTED, ha='left'
)

fig.subplots_adjust(left=0.13, right=0.96, top=0.83, bottom=0.17)

png_path = FIGURE_DIR / 'rice_supply_exposure_bubble_2024_2026.png'
svg_path = FIGURE_DIR / 'rice_supply_exposure_bubble_2024_2026.svg'
fig.savefig(png_path, dpi=300, bbox_inches='tight', facecolor=WHITE)
fig.savefig(svg_path, bbox_inches='tight', facecolor=WHITE)
plt.show()

print('PNG saved to:', png_path.resolve())
print('SVG saved to:', svg_path.resolve())

## 7. Export the chart dataset

The exported CSV keeps the exact values used for the bubble chart, including the descriptive reference group. Use this file for poster labels or later dashboard development.

In [ ]:
export_columns = [
    'Country', 'SSR_pct', 'Production_Shortfall_pct',
    'Beginning_Stock_to_Use_pct', 'HHI', 'HHI_10000',
    'effective_suppliers', 'top_supplier', 'top_supplier_share_pct',
    'import_tonnes', 'HHI_reference_group',
]

chart_data_path = DATA_OUTPUT_DIR / 'rice_supply_exposure_bubble_data_2024_2026.csv'
bubble[export_columns].sort_values('HHI_10000', ascending=False).to_csv(
    chart_data_path, index=False
)

display(bubble[export_columns].sort_values('HHI_10000', ascending=False))
print('Bubble-chart data saved to:', chart_data_path.resolve())

## Interpretation note

Read the chart as a comparison of exposure dimensions, not as a forecast. Higher and further right means that a country has both a larger domestic production gap and more concentrated supplier routes. Bubble size adds the scale of current import dependence.

The four countries are kept together because they are the complete set of assessed ASEAN members with a positive production shortfall and the bilateral-trade coverage used here. Thailand, Viet Nam, Cambodia, Indonesia, Lao PDR, and Myanmar are not added simply to fill the chart because their domestic-production positions answer a different question.